In [1]:
import json
import re
from google.cloud import storage

In [2]:
bucket_name = "rental_listing_data"

client = storage.Client()
bucket = client.bucket(bucket_name)

rows = []

pattern = re.compile(
    r"^(\d{4}-\d{2}-\d{2})_rentals_api_call$"
)

for blob in bucket.list_blobs():
    match = pattern.match(blob.name)

    if not match:
        continue

    snapshot_date = match.group(1)

    if snapshot_date < "2026-06-11":
        continue

    items = json.loads(blob.download_as_text())

    rows.append(
        {
            "snapshot_date": snapshot_date,
            "api_records": len(items),
            "distinct_ids": len(
                {
                    item.get("id")
                    for item in items
                    if item.get("id") is not None
                }
            ),
        }
    )

for row in sorted(rows, key=lambda x: x["snapshot_date"]):
    print(row)

{'snapshot_date': '2026-06-11', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-12', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-13', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-14', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-15', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-16', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-17', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-18', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-19', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-20', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-21', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-22', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-23', 'api_records': 500, 'distinct_ids': 500}
{'snapshot_date': '2026-06-24', 'api_records': 500,

In [4]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
url = "https://api.rentcast.io/v1/listings/rental/long-term"
rent_api_key = os.environ.get("RENALS_API_KEY")

params = {
    "city": "San Francisco",
    "state": "CA",
    "status": "Inactive",
    "limit": 500,
    "offset": 0,
    "includeTotalCount": "true",
}

headers = {
    "accept": "application/json",
    "X-Api-Key": rent_api_key,
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=60,
)

response.raise_for_status()

items = response.json()

print("Rows returned:", len(items))
print(
    "Total inactive available:",
    response.headers.get("X-Total-Count"),
)

print(items[0]["id"])
print(items[0]["listedDate"])
print(items[0]["removedDate"])
print(items[0].get("history"))

Rows returned: 500
Total inactive available: 37827
2485-Chestnut-St,-Apt-205,-San-Francisco,-CA-94123
2026-08-06T00:00:00.000Z
2026-08-07T00:00:00.000Z
{'2026-08-06': {'event': 'Rental Listing', 'price': 3495, 'listingType': 'Standard', 'listedDate': '2026-08-06T00:00:00.000Z', 'removedDate': '2026-08-07T00:00:00.000Z', 'daysOnMarket': 1}}


In [8]:
import pandas as pd

In [9]:
API_URL = "https://api.rentcast.io/v1/listings/rental/long-term"

HEADERS = {
    "accept": "application/json",
    "X-Api-Key": rent_api_key,
}


def fetch_all_rentals(status):
    params = {
        "city": "San Francisco",
        "state": "CA",
        "status": status,
        "limit": 500,
        "offset": 0,
        "includeTotalCount": "true",
    }

    all_items = []
    expected_total = None

    while True:
        response = requests.get(
            API_URL,
            headers=HEADERS,
            params=params,
            timeout=60,
        )

        response.raise_for_status()

        page = response.json()

        if expected_total is None:
            expected_total = int(
                response.headers["X-Total-Count"]
            )

            print(
                f"{status}: "
                f"{expected_total:,} total listings"
            )

        all_items.extend(page)

        print(
            f"{status}: "
            f"{len(all_items):,} / "
            f"{expected_total:,}"
        )

        if len(all_items) >= expected_total:
            break

        params["offset"] += 500

    return all_items

In [10]:
inactive_items = fetch_all_rentals(
    "Inactive"
)

active_items = fetch_all_rentals(
    "Active"
)

Inactive: 37,827 total listings
Inactive: 500 / 37,827
Inactive: 1,000 / 37,827
Inactive: 1,500 / 37,827
Inactive: 2,000 / 37,827
Inactive: 2,500 / 37,827
Inactive: 3,000 / 37,827
Inactive: 3,500 / 37,827
Inactive: 4,000 / 37,827
Inactive: 4,500 / 37,827
Inactive: 5,000 / 37,827
Inactive: 5,500 / 37,827
Inactive: 6,000 / 37,827
Inactive: 6,500 / 37,827
Inactive: 7,000 / 37,827
Inactive: 7,500 / 37,827
Inactive: 8,000 / 37,827
Inactive: 8,500 / 37,827
Inactive: 9,000 / 37,827
Inactive: 9,500 / 37,827
Inactive: 10,000 / 37,827
Inactive: 10,500 / 37,827
Inactive: 11,000 / 37,827
Inactive: 11,500 / 37,827
Inactive: 12,000 / 37,827
Inactive: 12,500 / 37,827
Inactive: 13,000 / 37,827
Inactive: 13,500 / 37,827
Inactive: 14,000 / 37,827
Inactive: 14,500 / 37,827
Inactive: 15,000 / 37,827
Inactive: 15,500 / 37,827
Inactive: 16,000 / 37,827
Inactive: 16,500 / 37,827
Inactive: 17,000 / 37,827
Inactive: 17,500 / 37,827
Inactive: 18,000 / 37,827
Inactive: 18,500 / 37,827
Inactive: 19,000 / 37,827
I

In [11]:
with open(
    "sf_rentals_inactive_rebuild.json",
    "w",
) as f:
    json.dump(
        inactive_items,
        f,
        indent=2,
    )


with open(
    "sf_rentals_active_rebuild.json",
    "w",
) as f:
    json.dump(
        active_items,
        f,
        indent=2,
    )

In [12]:
all_items = (
    inactive_items
    + active_items
)

df = pd.DataFrame(all_items)

In [13]:
import geopandas as gpd
import numpy as np
from shapely import wkt

In [14]:
def format_bucket_to_gdf(df):
    
    def dict_to_list(d):
        if not isinstance(d, dict): 
            return []
        return [{**value, 'event_date': key} for key, value in d.items()]
    
    df1 = df.drop(columns=['removedDate', 'listedDate', 'price'])
    gdf = gpd.GeoDataFrame(df1, geometry=gpd.points_from_xy(df1['longitude'], df1['latitude']), crs='EPSG:4326')

    gdf['history_list'] = gdf['history'].apply(dict_to_list)
    gdf_exploded = gdf.explode('history_list')
    history_expanded = pd.json_normalize(gdf_exploded['history_list']).set_index(gdf_exploded.index)
    final_gdf = pd.concat([gdf_exploded.drop(columns=['history', 'history_list']), history_expanded], axis=1)

    mask = ((final_gdf['longitude'] > -122.355) | (final_gdf['longitude'] < -122.517) | (final_gdf['latitude'] > 37.835) | (final_gdf['latitude'] < 37.704))
    if mask.any():
        geocodes = gpd.tools.geocode(final_gdf.loc[mask, 'formattedAddress'], provider='arcgis')
        final_gdf.loc[mask, 'geometry'] = geocodes['geometry']
    if mask.any:
        final_gdf = final_gdf.drop(final_gdf[mask].index)
    return final_gdf

In [48]:
gdf = format_bucket_to_gdf(df)

In [52]:
gdf[gdf['event'].isna()]

,id,formattedAddress,addressLine1,addressLine2,city,state,stateFips,zipCode,county,countyFips,...,listingOffice,hoa,geometry,event,price,listingType,listedDate,removedDate,daysOnMarket,event_date
13311,"1675-Clay-St,-Apt-8,-San-Francisco,-CA-94109","1675 Clay St, Apt 8, San Francisco, CA 94109",1675 Clay St,Apt 8,San Francisco,CA,06,94109,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.42048 37.79225),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13312,"310-6th-Ave,-Apt-15,-San-Francisco,-CA-94118","310 6th Ave, Apt 15, San Francisco, CA 94118",310 6th Ave,Apt 15,San Francisco,CA,06,94118,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.46406 37.78248),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13313,"1512-Great-Hwy,-Apt-1,-San-Francisco,-CA-94122","1512 Great Hwy, Apt 1, San Francisco, CA 94122",1512 Great Hwy,Apt 1,San Francisco,CA,06,94122,San Francisco,075,...,"{'name': 'Coldwell Banker Realty', 'phone': '4...",NaN,POINT (-122.50866 37.75812),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13314,"2619-Mission-St,-Unit-33,-San-Francisco,-CA-94110","2619 Mission St, Unit 33, San Francisco, CA 94110",2619 Mission St,Unit 33,San Francisco,CA,06,94110,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.41836 37.75504),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13315,"11-Dolores-St,-Apt-16,-San-Francisco,-CA-94103","11 Dolores St, Apt 16, San Francisco, CA 94103",11 Dolores St,Apt 16,San Francisco,CA,06,94103,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.42644 37.76888),NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37822,"1463-11th-Ave,-San-Francisco,-CA-94122","1463 11th Ave, San Francisco, CA 94122",1463 11th Ave,None,San Francisco,CA,06,94122,NaN,NaN,...,NaN,NaN,POINT (-122.46855 37.76091),NaN,NaN,NaN,NaN,NaN,NaN,NaN
37823,"275-Claremont-Blvd,-San-Francisco,-CA-94127","275 Claremont Blvd, San Francisco, CA 94127",275 Claremont Blvd,None,San Francisco,CA,06,94127,NaN,NaN,...,NaN,NaN,POINT (-122.46457 37.74088),NaN,NaN,NaN,NaN,NaN,NaN,NaN
37824,"3465-Sacramento-St,-Apt-4,-San-Francisco,-CA-9...","3465 Sacramento St, Apt 4, San Francisco, CA 9...",3465 Sacramento St,Apt 4,San Francisco,CA,06,94118,NaN,NaN,...,NaN,NaN,POINT (-122.44967 37.78767),NaN,NaN,NaN,NaN,NaN,NaN,NaN
37825,"940-Hayes-St,-Apt-16,-San-Francisco,-CA-94117","940 Hayes St, Apt 16, San Francisco, CA 94117",940 Hayes St,Apt 16,San Francisco,CA,06,94117,NaN,NaN,...,NaN,NaN,POINT (-122.43205 37.77601),NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
gdf.columns

Index(['id', 'formattedAddress', 'addressLine1', 'addressLine2', 'city',
       'state', 'stateFips', 'zipCode', 'county', 'countyFips', 'latitude',
       'longitude', 'propertyType', 'bedrooms', 'bathrooms', 'squareFootage',
       'status', 'listingType', 'createdDate', 'lastSeenDate', 'daysOnMarket',
       'yearBuilt', 'lotSize', 'mlsName', 'mlsNumber', 'listingAgent',
       'listingOffice', 'hoa', 'geometry', 'event', 'price', 'listingType',
       'listedDate', 'removedDate', 'daysOnMarket', 'event_date'],
      dtype='object')

In [54]:
gdf[gdf['listedDate'].isna()]

,id,formattedAddress,addressLine1,addressLine2,city,state,stateFips,zipCode,county,countyFips,...,listingOffice,hoa,geometry,event,price,listingType,listedDate,removedDate,daysOnMarket,event_date
13311,"1675-Clay-St,-Apt-8,-San-Francisco,-CA-94109","1675 Clay St, Apt 8, San Francisco, CA 94109",1675 Clay St,Apt 8,San Francisco,CA,06,94109,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.42048 37.79225),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13312,"310-6th-Ave,-Apt-15,-San-Francisco,-CA-94118","310 6th Ave, Apt 15, San Francisco, CA 94118",310 6th Ave,Apt 15,San Francisco,CA,06,94118,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.46406 37.78248),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13313,"1512-Great-Hwy,-Apt-1,-San-Francisco,-CA-94122","1512 Great Hwy, Apt 1, San Francisco, CA 94122",1512 Great Hwy,Apt 1,San Francisco,CA,06,94122,San Francisco,075,...,"{'name': 'Coldwell Banker Realty', 'phone': '4...",NaN,POINT (-122.50866 37.75812),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13314,"2619-Mission-St,-Unit-33,-San-Francisco,-CA-94110","2619 Mission St, Unit 33, San Francisco, CA 94110",2619 Mission St,Unit 33,San Francisco,CA,06,94110,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.41836 37.75504),NaN,NaN,NaN,NaN,NaN,NaN,NaN
13315,"11-Dolores-St,-Apt-16,-San-Francisco,-CA-94103","11 Dolores St, Apt 16, San Francisco, CA 94103",11 Dolores St,Apt 16,San Francisco,CA,06,94103,San Francisco,075,...,{'name': 'RentSFNow'},NaN,POINT (-122.42644 37.76888),NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37822,"1463-11th-Ave,-San-Francisco,-CA-94122","1463 11th Ave, San Francisco, CA 94122",1463 11th Ave,None,San Francisco,CA,06,94122,NaN,NaN,...,NaN,NaN,POINT (-122.46855 37.76091),NaN,NaN,NaN,NaN,NaN,NaN,NaN
37823,"275-Claremont-Blvd,-San-Francisco,-CA-94127","275 Claremont Blvd, San Francisco, CA 94127",275 Claremont Blvd,None,San Francisco,CA,06,94127,NaN,NaN,...,NaN,NaN,POINT (-122.46457 37.74088),NaN,NaN,NaN,NaN,NaN,NaN,NaN
37824,"3465-Sacramento-St,-Apt-4,-San-Francisco,-CA-9...","3465 Sacramento St, Apt 4, San Francisco, CA 9...",3465 Sacramento St,Apt 4,San Francisco,CA,06,94118,NaN,NaN,...,NaN,NaN,POINT (-122.44967 37.78767),NaN,NaN,NaN,NaN,NaN,NaN,NaN
37825,"940-Hayes-St,-Apt-16,-San-Francisco,-CA-94117","940 Hayes St, Apt 16, San Francisco, CA 94117",940 Hayes St,Apt 16,San Francisco,CA,06,94117,NaN,NaN,...,NaN,NaN,POINT (-122.43205 37.77601),NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
from google.cloud import bigquery 

In [56]:
def pull_bigquery_table(table_name: str):
    """Careful!! This pulls the ENTIRE table, so watch out!"""
    
    BQclient = bigquery.Client()
    query = f'''SELECT * 
            FROM `neighboorhood-nachos.neighborhood_livability_data.{table_name}`'''
    
    return BQclient.query(query).to_dataframe()

def format_bqtable_to_gdf(df):
    df['geometry'] = df['geometry'].apply(wkt.loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

def add_ids_to_gdf(gdf, neighborhoods, police_districts):
    gdf1 = gpd.sjoin(gdf, police_districts[['police_district_id', 'geometry']], how='left', predicate='within')
    gdf1.drop(columns=['index_right'], inplace=True)
    gdf2 = gpd.sjoin(gdf1, neighborhoods[['neighborhood_id', 'geometry']], how='left', predicate='within')

    gdf2['neighborhood_id'] = gdf2['neighborhood_id'].fillna(404)
    gdf2['police_district_id'] = gdf2['police_district_id'].fillna(404)

    return gdf2

def format_gdf_final_columns(gdf):
    gdf1 = gdf[['id', 'listedDate', 'removedDate', 'neighborhood_id', 'police_district_id', 
                'propertyType', 'bedrooms', 'bathrooms', 'squareFootage', 'status', 
                'price', 'formattedAddress', 'latitude', 'longitude', 'geometry']].copy()
    gdf1.columns = ['rentcast_id', 'listed_date', 'removed_date', 'neighborhood_id', 'police_district_id', 
                    'property_type', 'beds', 'baths', 'square_footage', 'status', 'price', 
                    'formatted_address', 'lat', 'long', 'geometry']
    
    gdf1['geometry'] = gdf1.geometry.to_wkt()

    timestamp_cols = ['listed_date', 'removed_date']
    for col in timestamp_cols:
        gdf1[col] = pd.to_datetime(gdf1[col]).dt.tz_convert('UTC')
    
    gdf1 = gdf1.astype(
    {'rentcast_id': str,
     'property_type': str,
     'beds': float,
     'baths': float,
     'square_footage': float,
     'status': str,
     'price': 'Int64',
     'formatted_address': str,
     'lat': float,
     'long': float})

    return gdf1

In [57]:
df_neighborhoods = pull_bigquery_table('neighborhoods')
gdf_neighborhoods = format_bqtable_to_gdf(df_neighborhoods)
gdf_neighborhoods.columns = ['neighborhood_id', 'name', 'geometry']

df_policedistricts = pull_bigquery_table('police_districts')
gdf_policedistricts = format_bqtable_to_gdf(df_policedistricts)
gdf_policedistricts.columns = ['police_district_id', 'name', 'geometry']

/opt/anaconda3/envs/env_transform_rentals/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/anaconda3/envs/env_transform_rentals/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [58]:
gdf = add_ids_to_gdf(gdf, gdf_neighborhoods, gdf_policedistricts)
gdf = format_gdf_final_columns(gdf)

/var/folders/06/wr5z_lvj1gg8xf40dsck2zdm0000gn/T/ipykernel_80756/1503128735.py:32: UserWarning: Geometry column does not contain geometry.
  gdf1['geometry'] = gdf1.geometry.to_wkt()


In [59]:
gdf = (
    gdf
    .sort_values(
        [
            "listed_date",
        ]
    )
    .drop_duplicates(
        subset=[
            "rentcast_id",
            "listed_date",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

In [60]:
before = len(gdf)

gdf = gdf.dropna(
    subset=[
        "rentcast_id",
        "listed_date",
    ]
).copy()

removed = before - len(gdf)

print(
    f"Dropped {removed:,} rows "
    "with missing rentcast_id or listed_date"
)

Dropped 24,491 rows with missing rentcast_id or listed_date


In [61]:
gdf.head()

,rentcast_id,listed_date,removed_date,neighborhood_id,police_district_id,property_type,beds,baths,square_footage,status,price,formatted_address,lat,long,geometry
0,"1515-15th-St,-Apt-407,-San-Francisco,-CA-94103",2016-10-28 00:00:00+00:00,2026-07-19 00:00:00+00:00,53,3,Condo,2.0,2.0,968.0,Active,5495,"1515 15th St, Apt 407, San Francisco, CA 94103",37.766574,-122.417981,POINT (-122.417981 37.766574)
1,"626-Pine-St,-Apt-203,-San-Francisco,-CA-94108",2021-12-13 00:00:00+00:00,2026-05-29 00:00:00+00:00,104,6,Apartment,0.0,1.0,NaN,Inactive,1750,"626 Pine St, Apt 203, San Francisco, CA 94108",37.791622,-122.406227,POINT (-122.406227 37.791622)
2,"724-35th-Ave,-Apt-2,-San-Francisco,-CA-94121",2022-02-22 00:00:00+00:00,2025-01-01 00:00:00+00:00,8,8,Apartment,2.0,1.0,1000.0,Inactive,2800,"724 35th Ave, Apt 2, San Francisco, CA 94121",37.775265,-122.494888,POINT (-122.494888 37.775265)
3,"106-Sargent-St,-San-Francisco,-CA-94132",2022-02-25 00:00:00+00:00,2026-03-03 00:00:00+00:00,65,10,Single Family,1.0,1.0,NaN,Inactive,1200,"106 Sargent St, San Francisco, CA 94132",37.716317,-122.463757,POINT (-122.463757 37.716317)
4,"2936-Cesar-Chavez,-San-Francisco,-CA-94110",2022-02-25 00:00:00+00:00,2025-05-28 00:00:00+00:00,53,3,Apartment,3.0,1.0,1850.0,Inactive,4000,"2936 Cesar Chavez, San Francisco, CA 94110",37.748528,-122.408440,POINT (-122.40844 37.748528)


In [62]:
# Remove unusable listing-history shells
before = len(gdf)

gdf = gdf.dropna(
    subset=[
        "rentcast_id",
        "listed_date",
    ]
).copy()

print(
    f"Dropped {before - len(gdf):,} rows "
    "without a listing date"
)


# Make sure we only have one row per listing episode
gdf = (
    gdf
    .drop_duplicates(
        subset=[
            "rentcast_id",
            "listed_date",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

print(
    f"Final reconstruction rows: {len(gdf):,}"
)

Dropped 0 rows without a listing date
Final reconstruction rows: 16,092


In [63]:
duplicates = gdf.duplicated(
    subset=[
        "rentcast_id",
        "listed_date",
    ],
    keep=False,
)

if duplicates.any():
    raise RuntimeError(
        f"Found {duplicates.sum():,} duplicate "
        "listing episodes."
    )

In [64]:
def create_reconstruction_table(
    source_table,
    reconstruction_table,
):
    client = bigquery.Client()

    query = f"""
        CREATE OR REPLACE TABLE
        `neighboorhood-nachos.neighborhood_livability_data.{reconstruction_table}`
        AS

        SELECT *
        FROM `neighboorhood-nachos.neighborhood_livability_data.{source_table}`

        WHERE FALSE
    """

    job = client.query(query)

    return job.result()

In [65]:
create_reconstruction_table(
    source_table="rental_listings",
    reconstruction_table="rental_listings_reconstruction",
)

In [66]:
def load_reconstruction(
    gdf,
    reconstruction_table,
):
    table = (
        "neighboorhood-nachos."
        "neighborhood_livability_data."
        f"{reconstruction_table}"
    )

    client = bigquery.Client()

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )

    job = client.load_table_from_dataframe(
        gdf,
        table,
        job_config=job_config,
    )

    result = job.result()

    print(
        f"Loaded {len(gdf):,} rows into "
        f"{reconstruction_table}"
    )

    return result

In [67]:
reconstruction_table = (
    "rental_listings_reconstruction"
)

load_reconstruction(
    gdf,
    reconstruction_table,
)

/opt/anaconda3/envs/env_transform_rentals/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 16,092 rows into rental_listings_reconstruction


LoadJob<project=neighboorhood-nachos, location=us-central1, id=d9356e6a-ed54-4039-931e-5ec4b39b1fa2>

In [68]:
client = bigquery.Client()

query = """
    SELECT COUNT(*) AS row_count
    FROM
    `neighboorhood-nachos.neighborhood_livability_data.rental_listings_reconstruction`
"""

result = client.query(query).to_dataframe()

print(result)

/opt/anaconda3/envs/env_transform_rentals/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   row_count
0      16092


In [69]:
len(gdf)

16092